# DSA5208 Assignment 1: Disconnected Secondaries and a Downed Primary

### Group Members

-

### What this notebook tests

- **Test A: disconnected secondaries.** One or two secondaries are frozen with `docker pause` while the client writes to the primary.
- **Test B: downed primary.** Both secondaries are cut off from replication, the client writes to the primary, and the primary is killed before the write can replicate. The old primary is then restarted and has to roll back.

Each test runs under both configs in `CONFIGS`. All `docker` commands run from Python, so the steps always happen in the right order.

**Requirements:** the notebook runs on the machine with Docker, containers are named `mongo1` to `mongo3`, and data lives in `/data/db` (the official image default). Failpoints need `enableTestCommands=1`, which your setup already has.

## Setup

In [ ]:
# Only run if needed
# %pip install pymongo

In [1]:
from pymongo import MongoClient, ReadPreference
from pymongo.errors import ExecutionTimeout, PyMongoError, WTimeoutError
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern

import subprocess
import time

## Connections

- `client` connects to the whole replica set and always sends writes to the current primary, even after a failover.
- `direct` holds one direct connection per node, used to read each node's own copy of the data and to set failpoints.

In [2]:
REPLICA_SET_URI = "mongodb://mongo1:27017,mongo2:27017,mongo3:27017/?replicaSet=dsa5208"
DB_NAME = "university"
NODES = ["mongo1", "mongo2", "mongo3"]

client = MongoClient(REPLICA_SET_URI, serverSelectionTimeoutMS=5000)

direct = {
    node: MongoClient(f"mongodb://{node}:27017/?directConnection=true",
                      serverSelectionTimeoutMS=3000)
    for node in NODES
}

## Consistency Configurations

In [3]:
CONFIGS = {
    "majority": {"write_concern": "majority", "read_concern": "majority"},
    "w1-local": {"write_concern": 1, "read_concern": "local"}
}

READ_TIMEOUT_MS = 3000   # how long a read may wait
WTIMEOUT_MS = 3000       # how long a write waits for confirmation from other nodes
CATCHUP_SECONDS = 2      # safety margin for nodes to catch up after reconnecting

## Helper Functions

### Collections and test data

In [4]:
def get_collection(collection_name, config_name, target="primary"):
    """Return a collection handle with the config's write and read concern applied."""
    config = CONFIGS[config_name]
    read_preference = {
        "primary": ReadPreference.PRIMARY,
        "secondary": ReadPreference.SECONDARY,
    }[target]

    db = client.get_database(
        DB_NAME,
        write_concern=WriteConcern(w=config["write_concern"], wtimeout=WTIMEOUT_MS),
        read_concern=ReadConcern(config["read_concern"]),
        read_preference=read_preference,
    )
    return db[collection_name]


# Baseline writes wait for ALL three nodes, so every node starts from the same state.
SETUP_WRITE_CONCERN = WriteConcern(w=3, wtimeout=30000)


def reset_document(collection_name, id_field, document):
    """Create or overwrite a test document and wait until every node has it."""
    db = client.get_database(DB_NAME, write_concern=SETUP_WRITE_CONCERN)
    db[collection_name].replace_one({id_field: document[id_field]}, document, upsert=True)


def make_student(student_id, name, **overrides):
    student = {
        "student_id": student_id,
        "student_name": name,
        "student_status": "Undergraduate",
        "seniority": "Year 1",
        "academic_load": "Full Time",
    }
    student.update(overrides)
    return student

def count_rollback_lines(node):
    """Count ROLLBACK entries in a node's Docker log."""
    logs = subprocess.run(["docker", "logs", node], capture_output=True, text=True)
    return (logs.stdout + logs.stderr).count("ROLLBACK")

### Reading, writing and printing

In [5]:
def read_field(collection, query, field):
    """Read one field through the replica-set client, using the collection's read concern."""
    try:
        doc = collection.find_one(query, max_time_ms=READ_TIMEOUT_MS)
    except ExecutionTimeout:
        return "TIMEOUT"
    except PyMongoError as error:
        return f"ERROR ({type(error).__name__})"
    return doc.get(field) if doc else "NOT FOUND"


def read_node(node, query, field):
    """Read one field from a single node's own copy of the data."""
    try:
        doc = direct[node][DB_NAME]["students"].find_one(query)
    except PyMongoError as error:
        return f"ERROR ({type(error).__name__})"
    return doc.get(field) if doc else "NOT FOUND"


def try_write(collection, query, update):
    """Run an update and report whether the write concern was satisfied."""
    try:
        collection.update_one(query, update)
        return "acknowledged"
    except WTimeoutError:
        return f"NOT confirmed within {WTIMEOUT_MS} ms"


def show(label, value):
    print(f"  {label:<30} {value}")


def header(title, config_name):
    config = CONFIGS[config_name]
    print(f"=== {title} | w={config['write_concern']}, "
          f"readConcern={config['read_concern']} ===")

### Controlling the nodes

In [6]:
def docker(*args):
    """Run a docker command from the notebook, e.g. docker("kill", "mongo1")."""
    subprocess.run(["docker", *args], check=True, capture_output=True)
    print(f"  [docker {' '.join(args)}]")


def disconnect_replication(node):
    direct[node].admin.command({"configureFailPoint": "stopReplProducer", "mode": "alwaysOn"})
    time.sleep(2)


def reconnect_replication(node):
    direct[node].admin.command({"configureFailPoint": "stopReplProducer", "mode": "off"})


def current_primary(timeout=60):
    """Wait until the replica set has a primary and return its name, e.g. "mongo1"."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            return client.admin.command("hello")["primary"].split(":")[0]
        except (PyMongoError, KeyError):
            time.sleep(1)
    raise RuntimeError("No primary was elected in time")


def wait_until_secondary(node, timeout=120):
    """Wait until a restarted node has rejoined as a secondary (after any rollback)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if direct[node].admin.command("hello").get("secondary"):
                return
        except PyMongoError:
            pass
        time.sleep(2)
    raise RuntimeError(f"{node} did not rejoin as a secondary in time")

### Cluster Status and Recovery

In [1]:
def show_members():
    """Print each member's state and whether it has applied everything the primary has."""
    members = client.admin.command("replSetGetStatus")["members"]
    primary = next((m for m in members if m["stateStr"] == "PRIMARY"), None)
  
    for m in members:
        print(m["name"], m["stateStr"], m.get("optimeDate"))


def restore_cluster():
    """
    Undo anything these tests change, in case a run was interrupted:
    unpause and start every container, then turn replication back on.
    Errors are ignored because most nodes will already be in the right state.
    """
    for node in NODES:
        subprocess.run(["docker", "unpause", node], capture_output=True)
        subprocess.run(["docker", "start", node], capture_output=True)
    for node in NODES:
        try:
            reconnect_replication(node)
        except PyMongoError:
            pass  # a freshly restarted node has no failpoints set anyway
    print("Primary:", current_primary())
    show_members()

## Experiment Functions

### Test A: disconnected secondaries

`docker pause` freezes a container without touching Docker networking. To the other nodes and the client, a paused node simply stops responding.

In [8]:
def run_disconnected_secondaries(config_name, how_many=1):
    header(f"Disconnected secondaries: {how_many}", config_name)
    students_primary = get_collection("students", config_name, "primary")

    primary = current_primary()
    paused = [node for node in NODES if node != primary][:how_many]

    student_id = f"S006-{config_name}-{how_many}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(student_id, "Robert Tan"))

    for node in paused:
        docker("pause", node)
    try:
        update = {"$set": {"student_status": "Graduate"}}
        show("Write result:", try_write(students_primary, query, update))
        show("Primary read:", read_field(students_primary, query, "student_status"))
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        for node in paused:
            docker("unpause", node)

    time.sleep(CATCHUP_SECONDS)
    for node in NODES:
        show(f"{node} after reconnecting:", read_node(node, query, "student_status"))

### Test B: downed primary

1. Both secondaries are cut off from replication with the `stopReplProducer` failpoint, so they stop receiving new writes. They stay reachable and can still elect a new primary.
2. The client writes to the primary, and we check which nodes have the write.
3. The primary is killed with `docker kill`, the secondaries are reconnected, and they elect a new primary.
4. The old primary is restarted. Its write is not on the new primary, so it must roll the write back to rejoin.

A throwaway write to `stall_marker` right after the failpoint is set absorbs any write that slips through before replication actually stops. It uses `w=1`, because a majority write would hang with both secondaries cut off.

In [9]:
def run_downed_primary(config_name):
    header("Downed primary", config_name)
    students_primary = get_collection("students", config_name, "primary")

    old_primary = current_primary()
    rollback_lines_before = count_rollback_lines(old_primary)
    secondaries = [node for node in NODES if node != old_primary]

    student_id = f"S005-{config_name}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(
        student_id, "Alan White", academic_history=["B.Sc. Computer Science"]
    ))

    # 1. Cut both secondaries off, so the next write only reaches the primary
    for node in secondaries:
        disconnect_replication(node)
    try:
        marker = client.get_database(DB_NAME, write_concern=WriteConcern(w=1))["stall_marker"]
        marker.insert_one({"at": time.time()})  # absorbs a write that may slip through
        time.sleep(2)

        # 2. Write, then check which nodes have it
        update = {"$push": {"academic_history": "M.Sc. Data Science"}}
        show("Write result:", try_write(students_primary, query, update))
        show("Primary read:", read_field(students_primary, query, "academic_history"))
        for node in NODES:
            show(f"{node} has:", read_node(node, query, "academic_history"))

        # 3. Kill the primary before the write can replicate
        docker("kill", old_primary)
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        for node in secondaries:
            reconnect_replication(node)

    new_primary = current_primary()
    show("New primary:", new_primary)

    # NEW: same client and read concern as "Primary read" before the kill
    show("Client read after failover:", read_field(students_primary, query, "academic_history"))

    # NEW: a later write from the same client
    students_primary.update_one(query, {"$push": {"academic_history": "PhD Information Systems"}})
    show("After writing PhD:", read_field(students_primary, query, "academic_history"))

    # 4. Restart the old primary; to rejoin, it must roll back the write
    docker("start", old_primary)

    # CHANGED: a w=3 write can only be confirmed once the old primary has
    # rolled back and caught up with the new primary's history
    catch_up = client.get_database(DB_NAME, write_concern=WriteConcern(w=3, wtimeout=120000))
    catch_up["stall_marker"].insert_one({"at": time.time()})

    show(f"{old_primary} after rejoining:", read_node(old_primary, query, "academic_history"))
    rolled_back = count_rollback_lines(old_primary) > rollback_lines_before
    show(f"{old_primary} rolled back:", "yes" if rolled_back else "no")

## Check the Cluster

`restore_cluster()` is safe to run at any time. Run it first, and again if any test is interrupted.

In [10]:
restore_cluster()

Primary: mongo1
  mongo1:27017   PRIMARY
  mongo2:27017   SECONDARY
  mongo3:27017   SECONDARY


## Test A1: One Disconnected Secondary

**What to look for:** whether the write is still confirmed and readable with one of three nodes unreachable, and whether the paused node catches up after reconnecting.

In [11]:
run_disconnected_secondaries("majority", how_many=1)

=== Disconnected secondaries: 1 | w=majority, readConcern=majority ===
  [docker pause mongo2]
  Write result:                  acknowledged
  Primary read:                  Graduate
  [docker unpause mongo2]
  mongo1 after reconnecting:     Graduate
  mongo2 after reconnecting:     Graduate
  mongo3 after reconnecting:     Graduate


In [12]:
run_disconnected_secondaries("w1-local", how_many=1)

=== Disconnected secondaries: 1 | w=1, readConcern=local ===
  [docker pause mongo2]
  Write result:                  acknowledged
  Primary read:                  Graduate
  [docker unpause mongo2]
  mongo1 after reconnecting:     Graduate
  mongo2 after reconnecting:     Graduate
  mongo3 after reconnecting:     Graduate


### Interpretation

_Compare the two configs here._

## Test A2: Two Disconnected Secondaries

**What to look for:** which config confirms the write when no other node can, what a read from the primary returns in each config, and whether the write survives once the secondaries reconnect.

Keep this quick: after about 10 seconds without contact with a majority, the primary steps down and all writes fail.

In [13]:
run_disconnected_secondaries("majority", how_many=2)

=== Disconnected secondaries: 2 | w=majority, readConcern=majority ===
  [docker pause mongo2]
  [docker pause mongo3]
  Write result:                  NOT confirmed within 3000 ms
  Primary read:                  Undergraduate
  [docker unpause mongo2]
  [docker unpause mongo3]
  mongo1 after reconnecting:     Graduate
  mongo2 after reconnecting:     Graduate
  mongo3 after reconnecting:     Graduate


In [14]:
run_disconnected_secondaries("w1-local", how_many=2)

=== Disconnected secondaries: 2 | w=1, readConcern=local ===
  [docker pause mongo2]
  [docker pause mongo3]
  Write result:                  acknowledged
  Primary read:                  Graduate
  [docker unpause mongo2]
  [docker unpause mongo3]
  mongo1 after reconnecting:     Graduate
  mongo2 after reconnecting:     Graduate
  mongo3 after reconnecting:     Graduate


### Interpretation

_Compare the two configs here._

## Test B: Downed Primary

**Check the run is valid first:** before the kill, only the primary's line should include M.Sc. If a secondary already has it, the write slipped through, so run `restore_cluster()` and try again.

**What to look for:** whether the client was told the write succeeded, whether any read showed it, and whether it survived the failover.

Rollback files build up across runs, so the second run also lists the first run's files.

In [15]:
run_downed_primary("w1-local")

=== Downed primary | w=1, readConcern=local ===
  Write result:                  acknowledged
  Primary read:                  ['B.Sc. Computer Science', 'M.Sc. Data Science']
  mongo1 has:                    ['B.Sc. Computer Science', 'M.Sc. Data Science']
  mongo2 has:                    ['B.Sc. Computer Science']
  mongo3 has:                    ['B.Sc. Computer Science']
  [docker kill mongo1]
  New primary:                   mongo2
  Client read after failover:    ['B.Sc. Computer Science']
  After writing PhD:             ['B.Sc. Computer Science', 'PhD Information Systems']
  [docker start mongo1]
  mongo1 after rejoining:        ['B.Sc. Computer Science', 'PhD Information Systems']
  mongo1 rolled back:            yes


In [17]:
show_members()

  mongo1:27017   (not reachable/healthy)
  mongo2:27017   PRIMARY
  mongo3:27017   SECONDARY


In [18]:
run_downed_primary("majority")

=== Downed primary | w=majority, readConcern=majority ===
  Write result:                  NOT confirmed within 3000 ms
  Primary read:                  ['B.Sc. Computer Science']
  mongo1 has:                    ['B.Sc. Computer Science']
  mongo2 has:                    ['B.Sc. Computer Science', 'M.Sc. Data Science']
  mongo3 has:                    ['B.Sc. Computer Science']
  [docker kill mongo2]
  New primary:                   mongo1
  Client read after failover:    ['B.Sc. Computer Science']
  After writing PhD:             ['B.Sc. Computer Science', 'PhD Information Systems']
  [docker start mongo2]
  mongo2 after rejoining:        ['B.Sc. Computer Science', 'PhD Information Systems']
  mongo2 rolled back:            yes


In [19]:
show_members()

  mongo1:27017   PRIMARY
  mongo2:27017   (not reachable/healthy)
  mongo3:27017   SECONDARY


### Interpretation

_Compare the two configs here._

## Clean Up

A different node may be primary now. Before returning to the stall experiments in the main notebook, check that mongo3 is not primary. If it is, run the cell below to step it down.

In [20]:
restore_cluster()
# If mongo3 is primary, uncomment:
# direct["mongo3"].admin.command("replSetStepDown", 60)

Primary: mongo1
  mongo1:27017   PRIMARY
  mongo2:27017   (not reachable/healthy)
  mongo3:27017   SECONDARY
